In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import *
from sklearn.impute import *
from sklearn.preprocessing import *
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import set_config


In [2]:
set_config(transform_output="pandas")

In [3]:
train_path = 'customer_churn_dataset-training-master.csv'
test_path = 'customer_churn_dataset-testing-master.csv'

In [4]:
df_train = pd.read_csv(train_path, index_col='CustomerID')
df_test = pd.read_csv(test_path, index_col='CustomerID')


In [5]:
df_train

,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Churn
CustomerID,,,,,,,,,,,
2.0,30.0,Female,39.0,14.0,5.0,18.0,Standard,Annual,932.00,17.0,1.0
3.0,65.0,Female,49.0,1.0,10.0,8.0,Basic,Monthly,557.00,6.0,1.0
4.0,55.0,Female,14.0,4.0,6.0,18.0,Basic,Quarterly,185.00,3.0,1.0
5.0,58.0,Male,38.0,21.0,7.0,7.0,Standard,Monthly,396.00,29.0,1.0
6.0,23.0,Male,32.0,20.0,5.0,8.0,Basic,Monthly,617.00,20.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
449995.0,42.0,Male,54.0,15.0,1.0,3.0,Premium,Annual,716.38,8.0,0.0
449996.0,25.0,Female,8.0,13.0,1.0,20.0,Premium,Annual,745.38,2.0,0.0
449997.0,26.0,Male,35.0,27.0,1.0,5.0,Standard,Quarterly,977.31,9.0,0.0


In [6]:
df_train.columns = [col.lower().replace(' ', '_') for col in df_train.columns]
df_test.columns = [col.lower().replace(' ', '_') for col in df_test.columns]


In [7]:
df_train.contract_length.value_counts()

contract_length
Annual       177198
Quarterly    176530
Monthly       87104
Name: count, dtype: int64

In [8]:
contract_order = ['Annual', 'Quarterly', 'Monthly']
sub_order = ['Premium','Standard', 'Basic']

In [9]:
df_test

,age,gender,tenure,usage_frequency,support_calls,payment_delay,subscription_type,contract_length,total_spend,last_interaction,churn
CustomerID,,,,,,,,,,,
1,22,Female,25,14,4,27,Basic,Monthly,598,9,1
2,41,Female,28,28,7,13,Standard,Monthly,584,20,0
3,47,Male,27,10,2,29,Premium,Annual,757,21,0
4,35,Male,9,12,5,17,Premium,Quarterly,232,18,0
5,53,Female,58,24,9,2,Standard,Annual,533,18,0
...,...,...,...,...,...,...,...,...,...,...,...
64370,45,Female,33,12,6,21,Basic,Quarterly,947,14,1
64371,37,Male,6,1,5,22,Standard,Annual,923,9,1
64372,25,Male,39,14,8,30,Premium,Monthly,327,20,1


In [10]:
df_train.dropna(axis='index', inplace=True)
df_test.dropna(axis='index', inplace=True)

In [11]:
X_train = df_train.drop(columns=['churn'])
y_train = df_train['churn']

In [12]:
X_test = df_test.drop(columns=['churn'])
y_test = df_test['churn']

In [13]:
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
cat_features

['gender', 'subscription_type', 'contract_length']

In [14]:
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_features

['age',
 'tenure',
 'usage_frequency',
 'support_calls',
 'payment_delay',
 'total_spend',
 'last_interaction']

#### Building a pipeline for testing different models

In [15]:
def build_preprocessor(scaler_type=None):
    cat_ct = ColumnTransformer([
        ('ohc', OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False), ['gender']),
        ('ordered_encoding', OrdinalEncoder(categories=[sub_order, contract_order]), cat_features[1:])
    ], remainder='passthrough')
    
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ct', cat_ct)
    ])
    
    num_steps = [('imputer', SimpleImputer(strategy='mean'))]
    if scaler_type == 'standard':
        num_steps.append(('scaler', StandardScaler()))
    elif scaler_type == 'minmax':
        num_steps.append(('scaler', MinMaxScaler()))
    num_pipe = Pipeline(steps=num_steps)
    
    preprocessor = ColumnTransformer([
        ('cat_pipe', cat_pipe, cat_features),
        ('num_pipe', num_pipe, num_features)
    ], remainder='drop')
    
    return preprocessor

base_models = {
    'LogisticRegressionCV': {
        'model': LogisticRegressionCV(Cs=[1.0], random_state=69, max_iter=1000),
        'scaler': 'standard'  
    },
    'RidgeClassifier': {
        'model': RidgeClassifier(alpha=1.0, random_state=69),
        'scaler': 'standard'
    },
    'KNeighborsClassifier': {
        'model': KNeighborsClassifier(n_neighbors=5),
        'scaler': 'standard'   
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=69),
        'scaler': None        
    },
    'AdaBoostClassifier': {
        'model': AdaBoostClassifier(n_estimators=100, learning_rate=1.0, random_state=69),
        'scaler': None
    },
    'XGBClassifier': {
        'model': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.3,
                               subsample=1.0, random_state=69, verbosity=0),
        'scaler': None        
    }
}


In [16]:
def build_pipeline(model_name, params):

    scaler_type = params.get('scaler_type', None)
    
    preprocessor = build_preprocessor(scaler_type)
    
    if model_name == 'LogisticRegressionCV':
        model = LogisticRegressionCV(Cs=[params['lr_Cs']], random_state=69, max_iter=1000)
    elif model_name == 'RidgeClassifier':
        model = RidgeClassifier(alpha=params['ridge_alpha'], random_state=69)
    elif model_name == 'KNeighborsClassifier':
        model = KNeighborsClassifier(n_neighbors=params['knn_n_neighbors'])
    elif model_name == 'RandomForestClassifier':
        model = RandomForestClassifier(
            n_estimators=params['rf_n_estimators'],
            max_depth=params['rf_max_depth'],
            random_state=69
        )
    elif model_name == 'AdaBoostClassifier':
        model = AdaBoostClassifier(
            n_estimators=params['ada_n_estimators'],
            learning_rate=params['ada_lr'],
            random_state=69
        )
    else:  # XGBClassifier
        model = XGBClassifier(
            n_estimators=params['xgb_n_estimators'],
            max_depth=params['xgb_max_depth'],
            learning_rate=params['xgb_learning_rate'],
            subsample=params['xgb_subsample'],
            random_state=69,
            verbosity=0
        )
    
    return Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

Scoring for models

In [17]:
def evaluate_model(model, model_scaler):
    preprocessor = build_preprocessor(model_scaler)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=3,
                             scoring='roc_auc', n_jobs=-1)
    return scores.mean()

baseline_scores = {}
for key, value in base_models.items():
    score = evaluate_model(model=value['model'], model_scaler=value['scaler'])
    baseline_scores[key] = score
    print(f"  {key}: ROC-AUC = {score:.4f}")

sorted_models = sorted(baseline_scores.items(), key=lambda x: x[1], reverse=True)
best_model_names = [name for name, _ in sorted_models[:3]]

  LogisticRegressionCV: ROC-AUC = 0.9448
  RidgeClassifier: ROC-AUC = 0.9448
  KNeighborsClassifier: ROC-AUC = 0.9865
  RandomForestClassifier: ROC-AUC = 1.0000
  AdaBoostClassifier: ROC-AUC = 0.9954
  XGBClassifier: ROC-AUC = 1.0000


In [18]:
def objective(trial, model_name):
    # Собираем словарь параметров для текущего испытания
    params = {}
    
    if model_name in ['LogisticRegressionCV', 'RidgeClassifier', 'KNeighborsClassifier']:
        params['scaler_type'] = trial.suggest_categorical('scaler_type', ['standard', 'minmax'])
    else:
        params['scaler_type'] = None
    
    if model_name == 'LogisticRegressionCV':
        params['lr_Cs'] = trial.suggest_float('lr_Cs', 1e-4, 10.0, log=True)
    elif model_name == 'RidgeClassifier':
        params['ridge_alpha'] = trial.suggest_float('ridge_alpha', 1e-3, 10.0, log=True)
    elif model_name == 'KNeighborsClassifier':
        params['knn_n_neighbors'] = trial.suggest_int('knn_n_neighbors', 3, 21, step=2)
    elif model_name == 'RandomForestClassifier':
        params['rf_n_estimators'] = trial.suggest_int('rf_n_estimators', 50, 500, step=50)
        params['rf_max_depth'] = trial.suggest_int('rf_max_depth', 3, 25)
    elif model_name == 'AdaBoostClassifier':
        params['ada_n_estimators'] = trial.suggest_int('ada_n_estimators', 50, 300, step=50)
        params['ada_lr'] = trial.suggest_float('ada_lr', 0.01, 1.0, log=True)
    else:  # XGBClassifier
        params['xgb_n_estimators'] = trial.suggest_int('xgb_n_estimators', 50, 500, step=50)
        params['xgb_max_depth'] = trial.suggest_int('xgb_max_depth', 3, 25)
        params['xgb_learning_rate'] = trial.suggest_float('xgb_learning_rate', 0.01, 0.5, log=True)
        params['xgb_subsample'] = trial.suggest_float('xgb_subsample', 0.8, 1.0)
    
    pipeline = build_pipeline(model_name, params)
    score = cross_val_score(pipeline, X_train, y_train, cv=3,
                            scoring='roc_auc', n_jobs=-1).mean()
    return score

In [20]:
best_params = {}
for model_name in best_model_names:
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, model_name),
                   n_trials=10, n_jobs=1,   
                   show_progress_bar=True)
    best_params[model_name] = study.best_params

[I 2026-06-17 21:18:36,134] A new study created in memory with name: no-name-cbe76109-8538-4a61-9855-e239b519c70e


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-06-17 21:18:38,113] Trial 0 finished with value: 0.9999993819912083 and parameters: {'xgb_n_estimators': 50, 'xgb_max_depth': 25, 'xgb_learning_rate': 0.31594267317725927, 'xgb_subsample': 0.8724449933212803}. Best is trial 0 with value: 0.9999993819912083.
[I 2026-06-17 21:18:48,179] Trial 1 finished with value: 0.9999993968943507 and parameters: {'xgb_n_estimators': 400, 'xgb_max_depth': 21, 'xgb_learning_rate': 0.1141384182377573, 'xgb_subsample': 0.9669846989566097}. Best is trial 1 with value: 0.9999993968943507.
[I 2026-06-17 21:19:00,973] Trial 2 finished with value: 0.9999995083849477 and parameters: {'xgb_n_estimators': 500, 'xgb_max_depth': 18, 'xgb_learning_rate': 0.03899425470704463, 'xgb_subsample': 0.9572197475406145}. Best is trial 2 with value: 0.9999995083849477.
[I 2026-06-17 21:19:07,682] Trial 3 finished with value: 0.9999992805618044 and parameters: {'xgb_n_estimators': 250, 'xgb_max_depth': 16, 'xgb_learning_rate': 0.12947119906986662, 'xgb_subsample': 0.8

[I 2026-06-17 21:19:36,925] A new study created in memory with name: no-name-ea3136f7-e09e-460a-9e09-1b799c13849a


[I 2026-06-17 21:19:36,923] Trial 9 finished with value: 0.9999995055552372 and parameters: {'xgb_n_estimators': 150, 'xgb_max_depth': 14, 'xgb_learning_rate': 0.08605803316169028, 'xgb_subsample': 0.9301417886457212}. Best is trial 2 with value: 0.9999995083849477.


...

[I 2026-06-17 21:19:55,077] Trial 0 finished with value: 0.9999997867655868 and parameters: {'rf_n_estimators': 100, 'rf_max_depth': 19}. Best is trial 0 with value: 0.9999997867655868.
[I 2026-06-17 21:21:12,652] Trial 1 finished with value: 0.9999997820808438 and parameters: {'rf_n_estimators': 450, 'rf_max_depth': 20}. Best is trial 0 with value: 0.9999997867655868.
[I 2026-06-17 21:21:30,482] Trial 2 finished with value: 0.9999997362709735 and parameters: {'rf_n_estimators': 100, 'rf_max_depth': 21}. Best is trial 0 with value: 0.9999997867655868.
[I 2026-06-17 21:21:57,454] Trial 3 finished with value: 0.9999997864511744 and parameters: {'rf_n_estimators': 150, 'rf_max_depth': 24}. Best is trial 0 with value: 0.9999997867655868.
[I 2026-06-17 21:22:13,051] Trial 4 finished with value: 0.9999986721740287 and parameters: {'rf_n_estimators': 100, 'rf_max_depth': 12}. Best is trial 0 with value: 0.9999997867655868.
[I 2026-06-17 21:22:33,453] Trial 5 finished with value: 0.99988262115

[I 2026-06-17 21:25:56,132] A new study created in memory with name: no-name-aadb803b-bfc6-48dc-b538-729c33f90995


[I 2026-06-17 21:25:56,130] Trial 9 finished with value: 0.9999997926765377 and parameters: {'rf_n_estimators': 500, 'rf_max_depth': 24}. Best is trial 9 with value: 0.9999997926765377.


  0%|          | 0/10 [00:00<?, ?it/s]

[I 2026-06-17 21:26:29,952] Trial 0 finished with value: 0.9949679848081924 and parameters: {'ada_n_estimators': 250, 'ada_lr': 0.5051647925649229}. Best is trial 0 with value: 0.9949679848081924.
[I 2026-06-17 21:27:10,140] Trial 1 finished with value: 0.9829552746706058 and parameters: {'ada_n_estimators': 300, 'ada_lr': 0.02929753927345021}. Best is trial 0 with value: 0.9949679848081924.
[I 2026-06-17 21:27:31,394] Trial 2 finished with value: 0.9832423821371252 and parameters: {'ada_n_estimators': 150, 'ada_lr': 0.09037445160166384}. Best is trial 0 with value: 0.9949679848081924.
[I 2026-06-17 21:28:12,173] Trial 3 finished with value: 0.9942217320550596 and parameters: {'ada_n_estimators': 300, 'ada_lr': 0.284457130886609}. Best is trial 0 with value: 0.9949679848081924.
[I 2026-06-17 21:28:52,843] Trial 4 finished with value: 0.9961556814361701 and parameters: {'ada_n_estimators': 300, 'ada_lr': 0.6847304181220517}. Best is trial 4 with value: 0.9961556814361701.
[I 2026-06-17 

In [ ]:
final_pipelines = {}
for model_name in best_model_names:
    params = best_params[model_name].copy()  # копируем, чтобы не испортить словарь
    
    # Для деревьев убираем scaler_type, если он там есть (он не нужен)
    if model_name in ['RandomForestClassifier', 'XGBClassifier']:
        params.pop('scaler_type', None)
    
    print(f"Обучаем {model_name} с параметрами: {params}")
    pipeline = build_pipeline(model_name, params)
    pipeline.fit(X_train, y_train)
    final_pipelines[model_name] = pipeline

# Делаем предсказания на тесте
print("\nПредсказания на X_test:")
for name, pipeline in final_pipelines.items():
    y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    print(f"   {name}: форма предсказаний {y_pred_proba.shape}")
    
    # Если есть истинные метки – вычисляем ROC-AUC
    if 'y_test' in locals():
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(y_test, y_pred_proba)
        print(f"      ROC-AUC на тесте: {auc:.4f}")


🚀 Обучаем XGBClassifier с параметрами: {'xgb_n_estimators': 500, 'xgb_max_depth': 18, 'xgb_learning_rate': 0.03899425470704463, 'xgb_subsample': 0.9572197475406145}
🚀 Обучаем RandomForestClassifier с параметрами: {'rf_n_estimators': 500, 'rf_max_depth': 24}
🚀 Обучаем AdaBoostClassifier с параметрами: {'ada_n_estimators': 300, 'ada_lr': 0.6847304181220517}

📊 Предсказания на X_test:
   XGBClassifier: форма предсказаний (64374,)
      ROC-AUC на тесте: 0.6938
   RandomForestClassifier: форма предсказаний (64374,)
      ROC-AUC на тесте: 0.5965
   AdaBoostClassifier: форма предсказаний (64374,)
      ROC-AUC на тесте: 0.7788


#### Data drift

So as we can see the accuracy on a test dataset is not so great for some of the best of our models. And why is that? Its because of the data drift in the datasets

In this article https://medium.com/@nursena_kok/from-89-to-58-how-data-drift-destroyed-my-model-and-how-i-caught-it-6f200266c8eb author faced the same issue with this dataset, and she explored this. She used KS test and PSI test and results shown that there are feature distribution drift and a slight target variable drift. PSI showed that some drifts are crucial 

#### The solution

The main solution to this is to train our model on a test (data with new distribution), so it can capture new behaviours of customers